# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/jh-emon002/flyrank-intern/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [25]:
%pip -q install duckdb huggingface_hub scikit-learn

In [26]:
import os
import duckdb
import pandas as pd

from huggingface_hub import HfApi

# Get HF token safely from Colab Secrets
try:
    from google.colab import userdata
    HF_TOKEN = userdata.get("HF_TOKEN")
except Exception:
    HF_TOKEN = os.environ.get("HF_TOKEN")

assert HF_TOKEN, "HF_TOKEN not found. Add it in Colab Secrets."

# Verify the token itself without printing it
who = HfApi(token=HF_TOKEN).whoami()
print("Authenticated as:", who["name"])

# DuckDB connection
con = duckdb.connect()

con.execute(
    f"CREATE OR REPLACE SECRET hf "
    f"(TYPE huggingface, TOKEN '{HF_TOKEN}')"
)

REL = "hf://datasets/FlyRank/internship-warehouse"

MARCH = (
    f"read_parquet("
    f"'{REL}/fact_content_daily_performance/month=2026-03/*.parquet'"
    f")"
)

print("Setup complete.")

Authenticated as: jh-emon002
Setup complete.


## 1. Unit of analysis + time window

1. **One row**: In the raw warehouse table, one row represents one reporting date × one pseudonymized client × one pseudonymized content item. For my Lane 2 analysis, I will aggregate those daily records so that one row in the final feature frame represents one content item at a single decision point.

2. **Source table**: I will use the March 2026 partition of `fact_content_daily_performance`.

3. **Time window**: March 1–15, 2026 is my feature window. March 16 is treated as the decision point. March 17–31 is my outcome window. Both feature and outcome windows therefore contain 15 days.

4. **What I predict/rank**: I will construct a temporary proxy called `is_declining_next15d`. For eligible pages, it is 1 when impressions in March 17–31 are more than 20% lower than impressions in March 1–15. A future model could estimate this decline risk and use the resulting score to help rank pages for human content review.

5. **Deliberate exclusion**: I will not use the June _sample table while designing this label because June is the final month of the panel. I will also exclude outcome-derived variables from the honest model features.

In [27]:
FEATURE_START = "2026-03-01"
FEATURE_END = "2026-03-15"

DECISION_DATE = "2026-03-16"

OUTCOME_START = "2026-03-17"
OUTCOME_END = "2026-03-31"

MIN_FEATURE_IMPRESSIONS = 100
DECLINE_RATIO_THRESHOLD = 0.80

print("Feature window :", FEATURE_START, "to", FEATURE_END)
print("Decision point :", DECISION_DATE)
print("Outcome window :", OUTCOME_START, "to", OUTCOME_END)
print("Minimum feature-window impressions:", MIN_FEATURE_IMPRESSIONS)
print("Decline threshold:", DECLINE_RATIO_THRESHOLD)

Feature window : 2026-03-01 to 2026-03-15
Decision point : 2026-03-16
Outcome window : 2026-03-17 to 2026-03-31
Minimum feature-window impressions: 100
Decline threshold: 0.8


## 2. Fields: feature / label / context / excluded

| Bucket            | Fields                                                                                                                 | Reason                                                                           |
| ----------------- | ---------------------------------------------------------------------------------------------------------------------- | -------------------------------------------------------------------------------- |
| **Feature**       | `impressions_pre15`, `ctr_pre15_pct`, `avg_position_pre15`, `position_volatility_pre15`, `days_with_impressions_pre15` | Calculated only from March 1–15, so knowable at the decision point               |
| **Label / proxy** | `is_declining_next15d`                                                                                                 | Outcome I am trying to predict                                                   |
| **Label source**  | `impressions_next15`, `decline_ratio`                                                                                  | Calculated from March 17–31 and used to construct the proxy                      |
| **Context**       | `client_hash_id`, `content_hash_id`, `report_date`, `gsc_data_available`                                               | Used for grouping, timing and availability checks, not learned as model features |
| **Excluded**      | `decline_ratio` from the honest model; June `_sample` from development                                                 | `decline_ratio` contains label information; June is the sealed final month       |


In [28]:
FEATURES = [
    "impressions_pre15",
    "ctr_pre15_pct",
    "avg_position_pre15",
    "position_volatility_pre15",
    "days_with_impressions_pre15",
]

TARGET = "is_declining_next15d"

print("Honest features:")
for col in FEATURES:
    print("-", col)

print("\nTarget:", TARGET)


Honest features:
- impressions_pre15
- ctr_pre15_pct
- avg_position_pre15
- position_volatility_pre15
- days_with_impressions_pre15

Target: is_declining_next15d


## 3. Verify it with queries (grain, counts, missing values, windows)

### Verification Query 1 — raw grain

The documented raw grain is one report_date × client_hash_id × content_hash_id. If this query returns zero rows, no duplicate combinations were found in the March slice.

In [29]:
grain_check = con.sql(f"""
    SELECT
        report_date,
        client_hash_id,
        content_hash_id,
        COUNT(*) AS n
    FROM {MARCH}
    GROUP BY
        report_date,
        client_hash_id,
        content_hash_id
    HAVING COUNT(*) > 1
    LIMIT 10
""").df()

print("Duplicate grain combinations found:", len(grain_check))
grain_check


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Duplicate grain combinations found: 0


,report_date,client_hash_id,content_hash_id,n


No duplicate `(report_date, client_hash_id, content_hash_id)` combinations were observed in this check, which supports the documented page-day grain.

### Verification Query 2 — size and date range

This query measures how many raw rows, clients and content items are present in the March partition and verifies the actual date boundaries.

In [30]:
slice_check = con.sql(f"""
    SELECT
        COUNT(*) AS row_count,
        COUNT(DISTINCT client_hash_id) AS clients,
        COUNT(DISTINCT content_hash_id) AS content_items,
        MIN(report_date) AS first_date,
        MAX(report_date) AS last_date
    FROM {MARCH}
""").df()

slice_check

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,row_count,clients,content_items,first_date,last_date
0,9841378,55,331437,2026-03-01,2026-03-31


The March slice contains 9841378 raw rows, covering 55 clients and 331437 content items. The measured date range is March 1, 2026 through March 31, 2026.

### Verification Query 3 — GSC availability

Search Console fields should only be treated as available when gsc_data_available `IS TRUE`. This query shows how many March rows survive that availability rule.

In [31]:
availability_check = con.sql(f"""
    SELECT
        COUNT(*) AS total_rows,

        COUNT(*) FILTER (
            WHERE gsc_data_available IS TRUE
        ) AS gsc_available_rows,

        ROUND(
            100.0 *
            COUNT(*) FILTER (
                WHERE gsc_data_available IS TRUE
            ) / COUNT(*),
            2
        ) AS gsc_available_pct

    FROM {MARCH}
""").df()

availability_check

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,total_rows,gsc_available_rows,gsc_available_pct
0,9841378,3611061,36.69


Of the 9841378 raw March rows, 3611061 have `gsc_data_available` `IS TRUE`, corresponding to 36.69% of the slice.

### Feature Building Query - one row per content item


In [32]:
feature_frame = con.sql(f"""
WITH page_windows AS (

    SELECT
        client_hash_id,
        content_hash_id,

        -- How many usable GSC days exist in each window?
        COUNT(DISTINCT CASE
            WHEN report_date BETWEEN DATE '{FEATURE_START}'
                                 AND DATE '{FEATURE_END}'
             AND gsc_data_available IS TRUE
            THEN report_date
        END) AS feature_days_available,

        COUNT(DISTINCT CASE
            WHEN report_date BETWEEN DATE '{OUTCOME_START}'
                                 AND DATE '{OUTCOME_END}'
             AND gsc_data_available IS TRUE
            THEN report_date
        END) AS outcome_days_available,

        -- FEATURE 1: impressions
        SUM(CASE
            WHEN report_date BETWEEN DATE '{FEATURE_START}'
                                 AND DATE '{FEATURE_END}'
             AND gsc_data_available IS TRUE
            THEN gsc_impressions
            ELSE 0
        END) AS impressions_pre15,

        -- Used to construct FEATURE 2: CTR
        SUM(CASE
            WHEN report_date BETWEEN DATE '{FEATURE_START}'
                                 AND DATE '{FEATURE_END}'
             AND gsc_data_available IS TRUE
            THEN gsc_clicks
            ELSE 0
        END) AS clicks_pre15,

        -- FEATURE 3: impression-weighted average position
        SUM(CASE
            WHEN report_date BETWEEN DATE '{FEATURE_START}'
                                 AND DATE '{FEATURE_END}'
             AND gsc_data_available IS TRUE
            THEN gsc_avg_position * gsc_impressions
            ELSE 0
        END)
        /
        NULLIF(
            SUM(CASE
                WHEN report_date BETWEEN DATE '{FEATURE_START}'
                                     AND DATE '{FEATURE_END}'
                 AND gsc_data_available IS TRUE
                THEN gsc_impressions
                ELSE 0
            END),
            0
        ) AS avg_position_pre15,

        -- FEATURE 4: daily position volatility
        STDDEV_POP(CASE
            WHEN report_date BETWEEN DATE '{FEATURE_START}'
                                 AND DATE '{FEATURE_END}'
             AND gsc_data_available IS TRUE
             AND gsc_impressions > 0
            THEN gsc_avg_position
        END) AS position_volatility_pre15,

        -- FEATURE 5: number of days with impressions
        COUNT(DISTINCT CASE
            WHEN report_date BETWEEN DATE '{FEATURE_START}'
                                 AND DATE '{FEATURE_END}'
             AND gsc_data_available IS TRUE
             AND gsc_impressions > 0
            THEN report_date
        END) AS days_with_impressions_pre15,

        -- FUTURE / OUTCOME DATA — NOT A FEATURE
        SUM(CASE
            WHEN report_date BETWEEN DATE '{OUTCOME_START}'
                                 AND DATE '{OUTCOME_END}'
             AND gsc_data_available IS TRUE
            THEN gsc_impressions
            ELSE 0
        END) AS impressions_next15

    FROM {MARCH}

    GROUP BY
        client_hash_id,
        content_hash_id
)

SELECT
    client_hash_id,
    content_hash_id,

    impressions_pre15,

    100.0 * clicks_pre15
        / NULLIF(impressions_pre15, 0)
        AS ctr_pre15_pct,

    avg_position_pre15,
    COALESCE(position_volatility_pre15, 0)
        AS position_volatility_pre15,

    days_with_impressions_pre15,

    -- label sources
    impressions_next15,

    1.0 * impressions_next15
        / NULLIF(impressions_pre15, 0)
        AS decline_ratio,

    CASE
        WHEN impressions_next15
             < {DECLINE_RATIO_THRESHOLD} * impressions_pre15
        THEN 1
        ELSE 0
    END AS is_declining_next15d

FROM page_windows

WHERE
    feature_days_available = 15
    AND outcome_days_available = 15
    AND impressions_pre15 >= {MIN_FEATURE_IMPRESSIONS}
""").df()

print("Feature frame shape:", feature_frame.shape)

feature_frame.head(10)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Feature frame shape: (58097, 10)


,client_hash_id,content_hash_id,impressions_pre15,ctr_pre15_pct,avg_position_pre15,position_volatility_pre15,days_with_impressions_pre15,impressions_next15,decline_ratio,is_declining_next15d
0,client_73cda7b4e4f265ea,content_7a105f548d9c6916,4173.0,0.143781,6.265037,0.986143,15,2033.0,0.487179,1
1,client_73cda7b4e4f265ea,content_a3ea9792f793ec72,245.0,0.000000,4.085714,2.826336,15,195.0,0.795918,1
2,client_73cda7b4e4f265ea,content_36c36abc7650d7af,3705.0,0.080972,6.297706,1.280020,15,1726.0,0.465857,1
3,client_73cda7b4e4f265ea,content_a7da352b73b02668,2440.0,0.327869,7.370902,0.751107,15,2440.0,1.000000,0
4,client_73cda7b4e4f265ea,content_1855a661b4d36130,240.0,0.416667,3.770833,2.042009,15,165.0,0.687500,1
5,client_73cda7b4e4f265ea,content_5d412fba6e1a2582,131.0,0.000000,9.809160,3.954859,15,74.0,0.564885,1
6,client_73cda7b4e4f265ea,content_22c063002b7c1caf,172.0,0.000000,7.755814,4.714287,15,133.0,0.773256,1
7,client_73cda7b4e4f265ea,content_aafb2ab7e5fc80d0,3104.0,0.515464,5.542848,0.872418,15,4369.0,1.407539,0
8,client_73cda7b4e4f265ea,content_20403327d8d9374c,1294.0,0.386399,6.990726,2.287707,15,2058.0,1.590417,0
9,client_73cda7b4e4f265ea,content_f8df6b20d18c4374,527.0,0.189753,7.387097,1.851645,15,231.0,0.438330,1


`impressions_pre15` — Knowable at the decision moment because it uses only Search Console impressions recorded from March 1–15.

`ctr_pre15_pct` — Knowable at the decision moment because both its clicks and impressions were observed during March 1–15.

`avg_position_pre15` — Knowable at the decision moment because it summarizes only feature-window ranking observations.

`position_volatility_pre15` — Knowable at the decision moment because every daily position used in the calculation occurred before March 16.

`days_with_impressions_pre15` — Knowable at the decision moment because it only counts March 1–15 days with observed impressions.

In [33]:
print("Eligible content items:", len(feature_frame))

print(
    "Declining proxy positives:",
    feature_frame["is_declining_next15d"].sum()
)

print(
    "Proxy-positive rate:",
    f"{feature_frame['is_declining_next15d'].mean():.1%}"
)

feature_frame[[
    "content_hash_id",
    "impressions_pre15",
    "impressions_next15",
    "decline_ratio",
    "is_declining_next15d"
]].head(10)

Eligible content items: 58097
Declining proxy positives: 20036
Proxy-positive rate: 34.5%


,content_hash_id,impressions_pre15,impressions_next15,decline_ratio,is_declining_next15d
0,content_7a105f548d9c6916,4173.0,2033.0,0.487179,1
1,content_a3ea9792f793ec72,245.0,195.0,0.795918,1
2,content_36c36abc7650d7af,3705.0,1726.0,0.465857,1
3,content_a7da352b73b02668,2440.0,2440.0,1.000000,0
4,content_1855a661b4d36130,240.0,165.0,0.687500,1
5,content_5d412fba6e1a2582,131.0,74.0,0.564885,1
6,content_22c063002b7c1caf,172.0,133.0,0.773256,1
7,content_aafb2ab7e5fc80d0,3104.0,4369.0,1.407539,0
8,content_20403327d8d9374c,1294.0,2058.0,1.590417,0
9,content_f8df6b20d18c4374,527.0,231.0,0.438330,1


### Deliberate leakage experiment

`decline_ratio` is calculated using outcome-window impressions and directly determines `is_declining_next15d`. It must therefore never be included in the honest feature set. I will add it once deliberately to demonstrate the effect of target leakage.

In [34]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import accuracy_score

model_df = feature_frame.dropna(
    subset=FEATURES + [TARGET]
).copy()

X = model_df[FEATURES]
y = model_df[TARGET]

groups = model_df["client_hash_id"]

splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.25,
    random_state=42
)

train_idx, test_idx = next(
    splitter.split(X, y, groups=groups)
)

X_train = X.iloc[train_idx]
X_test = X.iloc[test_idx]

y_train = y.iloc[train_idx]
y_test = y.iloc[test_idx]

honest_model = DecisionTreeClassifier(
    max_depth=4,
    min_samples_leaf=20,
    random_state=42
)

honest_model.fit(X_train, y_train)

honest_pred = honest_model.predict(X_test)

honest_accuracy = accuracy_score(
    y_test,
    honest_pred
)

print("Proxy positive rate:", round(y.mean(), 3))
print("Honest quick accuracy:", round(honest_accuracy, 3))

Proxy positive rate: 0.345
Honest quick accuracy: 0.654


In [35]:
LEAKY_FEATURES = FEATURES + ["decline_ratio"]

X_leaky = model_df[LEAKY_FEATURES]

X_train_leaky = X_leaky.iloc[train_idx]
X_test_leaky = X_leaky.iloc[test_idx]

leaky_model = DecisionTreeClassifier(
    max_depth=4,
    min_samples_leaf=20,
    random_state=42
)

leaky_model.fit(
    X_train_leaky,
    y_train
)

leaky_pred = leaky_model.predict(
    X_test_leaky
)

leaky_accuracy = accuracy_score(
    y_test,
    leaky_pred
)

print("Honest quick accuracy:", round(honest_accuracy, 3))
print("Leaky quick accuracy: ", round(leaky_accuracy, 3))

Honest quick accuracy: 0.654
Leaky quick accuracy:  1.0


In [36]:
FINAL_FEATURES = FEATURES.copy()

assert "decline_ratio" not in FINAL_FEATURES
assert "impressions_next15" not in FINAL_FEATURES

print("Final honest feature set:")
for feature in FINAL_FEATURES:
    print("-", feature)

print("\nHonest score retained:", round(honest_accuracy, 3))

Final honest feature set:
- impressions_pre15
- ctr_pre15_pct
- avg_position_pre15
- position_volatility_pre15
- days_with_impressions_pre15

Honest score retained: 0.654


Adding `decline_ratio` caused the quick validation score to increase sharply because that variable is calculated from the same future impressions that define `is_declining_next15d`. The model was therefore receiving label-derived information rather than learning a useful predictive relationship. I removed decline_ratio and retain the honest score as the meaningful result. This quick accuracy is only used for the leakage demonstration; my lane remains a ranking/scoring problem whose eventual decision-facing evaluation should focus on the usefulness of the top-ranked review queue.

## 4. Data limits

**Limitation — coverage and selection bias**

This feature frame requires all 15 feature-window days, all 15 outcome-window days, and at least 100 impressions before the decision point. It therefore represents established, sufficiently visible pages better than newly registered or very low-volume content. The warehouse is also an unbalanced panel, so client histories do not all begin at the same time. Results from this March slice should therefore be treated as directional decision-support rather than evidence that the same relationships hold for every FlyRank content item.

In [37]:
march_content_items = int(
    slice_check.loc[0, "content_items"]
)

eligible_content_items = len(feature_frame)

retention_pct = (
    100 * eligible_content_items
    / march_content_items
)

print("March content items:", f"{march_content_items:,}")
print("Eligible feature-frame items:", f"{eligible_content_items:,}")
print("Retained after eligibility rules:", f"{retention_pct:.1f}%")


March content items: 331,437
Eligible feature-frame items: 58,097
Retained after eligibility rules: 17.5%


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.